# Day 17: DevOps and Cloud Fundamentals
**A local CI simulator, DevSecOps scans (SAST, dependencies, secrets), pipeline YAML with a security stage, and cloud model practice**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup
We cannot create cloud resources from this notebook, so we simulate a CI pipeline locally and prepare the files you push to Azure DevOps in the browser.

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q pytest ruff bandit pip-audit pyyaml openai tiktoken scikit-learn

# ===== Option B: VS Code / local =====
# pip install pytest ruff bandit pip-audit pyyaml openai tiktoken scikit-learn   (in your activated venv)
import os
os.makedirs("ci_demo", exist_ok=True)
os.chdir("ci_demo")
print("Working in", os.getcwd())

## 2. A tiny project to build and test

In [ ]:
%%writefile calc.py
def add(a, b):
    return a + b


def discount(price, pct):
    if not 0 <= pct <= 100:
        raise ValueError("pct must be 0 to 100")
    return round(price * (1 - pct / 100), 2)

In [ ]:
%%writefile test_calc.py
import pytest

from calc import add, discount


def test_add():
    assert add(2, 3) == 5


def test_discount():
    assert discount(200, 10) == 180.0


def test_bad_discount():
    with pytest.raises(ValueError):
        discount(100, 120)

In [ ]:
%%writefile requirements.txt
fastapi==0.115.0
pyyaml==6.0.2

## 3. A local CI simulator
Each step is a shell command. The pipeline stops at the first failure, just like Azure Pipelines.

In [ ]:
import subprocess, time, zipfile, glob, shutil

def run_pipeline(steps):
    shutil.rmtree("__pycache__", ignore_errors=True)   # a CI agent always starts clean, so do we
    env = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1"}
    start = time.perf_counter()
    for i, (name, cmd) in enumerate(steps, 1):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
        status = "OK" if r.returncode == 0 else "FAILED"
        print(f"[{i}/{len(steps)}] {name:<14} ... {status}")
        if r.returncode != 0:
            print((r.stdout + r.stderr)[-800:])
            print("Pipeline stopped. Fix the problem and push again.")
            return False
    print(f"Pipeline succeeded in {time.perf_counter() - start:.1f} s")
    return True

def package():
    with zipfile.ZipFile("app.zip", "w") as z:
        for f in glob.glob("*.py") + ["requirements.txt"]:
            if not f.startswith("test_"):
                z.write(f)
    return "app.zip"

STEPS = [
    ("Lint", "ruff check --isolated calc.py test_calc.py"),
    ("Unit tests", "python -m pytest -q test_calc.py"),
]
if run_pipeline(STEPS):
    print("Artifact:", package())

### Break it on purpose
This cell introduces a bug, runs the pipeline (it fails), then restores the file.

In [ ]:
good = open("calc.py").read()
open("calc.py", "w").write(good.replace("return a + b", "return a - b"))
run_pipeline(STEPS)
open("calc.py", "w").write(good)
print("\nBug fixed. Running again:")
_ = run_pipeline(STEPS)

### TODO 1: Add a step that fails if any Python file is longer than 200 lines, and see it pass.

In [ ]:
# TODO: write your code here

## 4. NEW (v4): DevSecOps: SAST, dependency and secret scanning
A small report module with three common problems. Imagine Codex wrote it quickly. The scans find what a quick read misses.

In [ ]:
%%writefile report.py
import subprocess

DB_PASSWORD = "changeme123"          # hard-coded secret (demo only)


def export(name):
    subprocess.run(f"zip {name}.zip data", shell=True)


def total(expr):
    return eval(expr)

In [ ]:
# SAST with bandit: -ll shows medium and high severity, -f txt for readable output
!bandit -q report.py || true

In [ ]:
import re

SECRET_PATTERNS = [
    re.compile(r"(password|passwd|secret|api_key|token)\s*=\s*['\"][^'\"]{6,}['\"]", re.I),
    re.compile(r"-----BEGIN (RSA |EC )?PRIVATE KEY-----"),
]

def scan_secrets(paths):
    findings = []
    for path in paths:
        for n, line in enumerate(open(path), 1):
            if any(p.search(line) for p in SECRET_PATTERNS):
                findings.append((path, n, line.strip()[:60]))
    return findings

for f in scan_secrets(glob.glob("*.py")):
    print("SECRET?", f)

In [ ]:
# Dependency scanning needs the internet to download the vulnerability database.
!pip-audit -r requirements.txt || true

### Fix, then re-scan
Move the secret to an environment variable, pass arguments as a list, and replace `eval` with a safe parser.

In [ ]:
%%writefile report.py
import ast
import os
import subprocess

DB_PASSWORD = os.environ.get("DB_PASSWORD", "")   # read from the environment or a secret store


def export(name):
    subprocess.run(["zip", f"{name}.zip", "data"], check=False)


def total(expr):
    tree = ast.parse(expr, mode="eval")
    if not all(isinstance(n, (ast.Expression, ast.BinOp, ast.Constant, ast.Add, ast.Sub, ast.Mult, ast.Div))
               for n in ast.walk(tree)):
        raise ValueError("only arithmetic is allowed")
    return eval(compile(tree, "<expr>", "eval"), {"__builtins__": {}})  # nosec B307: input checked above

In [ ]:
!bandit -q -ll report.py && echo "bandit: no medium or high issues"
print("secrets:", scan_secrets(["report.py"]) or "none found")

Notice the `# nosec` comment: it records **why** a finding is accepted. Never add it just to turn CI green. The remaining low-severity note on `subprocess` is reviewed and accepted.

### Explain a finding in plain English
Useful when you write the vulnerability report. Runs offline too.

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

_ENC = []

def count_tokens(text):
    if not _ENC:
        try:
            import tiktoken
            _ENC.append(tiktoken.get_encoding("o200k_base"))
        except Exception:
            _ENC.append(None)                        # no tiktoken or no download: use the estimate
    if _ENC[0] is not None:
        return len(_ENC[0].encode(text))
    return max(1, round(len(text) / 4))              # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

In [ ]:
finding = "B602 subprocess_popen_with_shell_equals_true, severity High, report.py line 7"
print(llm(f"Explain this bandit finding to a junior developer in two sentences, then give the fix: {finding}"))

## 5. The pipeline file for Azure DevOps, with a security stage

In [ ]:
%%writefile azure-pipelines.yml
trigger:
  - main
pr:
  - main

pool:
  vmImage: ubuntu-latest

stages:
  - stage: Build
    jobs:
      - job: test
        steps:
          - task: UsePythonVersion@0
            inputs:
              versionSpec: "3.12"
            displayName: Use Python 3.12
          - script: pip install -r requirements.txt pytest ruff
            displayName: Install dependencies
          - script: ruff check .
            displayName: Lint
          - script: python -m pytest -q --junitxml=test-results.xml
            displayName: Run tests
          - task: PublishTestResults@2
            condition: always()
            inputs:
              testResultsFiles: test-results.xml
            displayName: Publish test results

  - stage: Security
    dependsOn: Build
    jobs:
      - job: scan
        steps:
          - script: pip install bandit pip-audit
            displayName: Install scanners
          - script: bandit -r . -ll
            displayName: SAST (bandit)
          - script: pip-audit -r requirements.txt
            displayName: Dependency scan (pip-audit)

In [ ]:
import yaml
cfg = yaml.safe_load(open("azure-pipelines.yml"))
print("Trigger:", cfg["trigger"], "| PR:", cfg["pr"])
print("Agent:", cfg["pool"]["vmImage"])
for stage in cfg["stages"]:
    steps = stage["jobs"][0]["steps"]
    print(f"Stage {stage['stage']}:")
    for i, step in enumerate(steps, 1):
        kind = "task" if "task" in step else "script"
        print(f"   {i}. {step['displayName']}  ({kind})")

**TODO (SmartDesk step 15):** add a secret-scanning step to the Security stage and explain each stage in one sentence.

## 6. Cloud models and migration paths: quick check

In [ ]:
scenarios = {
    "Legacy app that needs a custom Windows driver": "IaaS",
    "FastAPI service the team just wants to deploy": "PaaS",
    "Company email and documents": "SaaS",
    "Managed relational database nobody wants to patch": "PaaS",
    "Full control of a GPU machine with custom drivers": "IaaS",
}
my_answers = {   # TODO: change these to your own answers, then run
    "Legacy app that needs a custom Windows driver": "IaaS",
    "FastAPI service the team just wants to deploy": "PaaS",
    "Company email and documents": "SaaS",
    "Managed relational database nobody wants to patch": "IaaS",
    "Full control of a GPU machine with custom drivers": "IaaS",
}
score = sum(my_answers[s] == m for s, m in scenarios.items())
for s, m in scenarios.items():
    mark = "correct" if my_answers[s] == m else f"expected {m}"
    print(f"- {s}: {my_answers[s]} ({mark})")
print(f"Score: {score}/{len(scenarios)}")

PATHS = ["rehost", "replatform", "refactor", "rearchitect", "rebuild", "replace"]
migrations = {"Old ticketing server, no time to change code": "rehost",
              "Same app, but move its database to Azure SQL": "replatform",
              "Rewrite the reporting tool as FastAPI + React": "rebuild",
              "Switch to a SaaS helpdesk product": "replace"}
for case, path in migrations.items():
    print(f"{path:<11} (step {PATHS.index(path) + 1} of 6)  {case}")

## 7. Azure in the browser
Open portal.azure.com with the program subscription. Create a resource group named `rg-day17-yourname` in the region your trainer gives you, add tags `owner` and `project`, look at the Overview page, then **delete it**. Then open dev.azure.com, create a user story in Boards and a new pipeline from the YAML above. No local Azure tools are needed.

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)